In [1]:
import pandas as pd
import numpy as np

In [2]:
df=pd.read_excel(r"C:\Users\samru\OneDrive\Desktop\SIH_2025\AI-Pollution-Forecast-and-Policy-Dashboard\ML\Raw\Air_Quality_Data\AQI_daily_2023_Burari_Crossing_Delhi_IMD_2023.xlsx")

In [8]:
df.head()

,Day,January,February,March,April,May,June,July,August,September,October,November,December
0,1,290.0,127.0,NaN,154.0,66.0,82.0,NaN,NaN,144.0,156.0,353.0,345.0
1,2,340.0,189.0,216.0,125.0,67.0,99.0,49.0,NaN,149.0,160.0,397.0,337.0
2,3,359.0,210.0,136.0,141.0,96.0,101.0,92.0,NaN,141.0,171.0,470.0,300.0
3,4,318.0,266.0,NaN,89.0,164.0,NaN,128.0,NaN,139.0,175.0,392.0,276.0
4,5,341.0,227.0,NaN,127.0,152.0,NaN,76.0,NaN,122.0,187.0,NaN,235.0


In [3]:
df_cleaned = df.drop_duplicates(keep='first')
df_cleaned.shape


(41, 13)

In [5]:
# Replace 'NA', empty strings, and explicit NaNs with np.nan for consistency
df_cleaned = df_cleaned.replace('NA', np.nan)
df_cleaned = df_cleaned.replace(r'^\s*$', np.nan, regex=True)

# Convert all columns except 'Day' to numeric
for col in df_cleaned.columns:
    if col != 'Day':
        df_cleaned[col] = pd.to_numeric(df_cleaned[col], errors='coerce')

# Fill missing values with the mean of each column
df_filled = df_cleaned.fillna(df_cleaned.mean(numeric_only=True))


In [6]:
# Define a function for outlier handling
def handle_outliers_iqr(df):
    for col in df.columns:
        if col != 'Day' and df[col].dtype != 'O':
            Q1 = df[col].quantile(0.25)
            Q3 = df[col].quantile(0.75)
            IQR = Q3 - Q1
            lower = Q1 - 1.5 * IQR
            upper = Q3 + 1.5 * IQR
            mean_val = df[col].mean()
            # Replace outliers with mean
            df[col] = np.where((df[col] < lower) | (df[col] > upper), mean_val, df[col])
    return df

df_no_outliers = handle_outliers_iqr(df_filled.copy())


In [7]:
# Drop non-feature rows if present, and reset index
df_ml_ready = df_no_outliers.copy()

# If your dataset contains summary/statistical rows (not data for 'Day'), remove them
df_ml_ready = df_ml_ready[df_ml_ready['Day'].apply(lambda x: str(x).isdigit())]
df_ml_ready = df_ml_ready.reset_index(drop=True)

# Optional: Convert 'Day' to int if needed
df_ml_ready['Day'] = df_ml_ready['Day'].astype(int)

df_ml_ready.head()


,Day,January,February,March,April,May,June,July,August,September,October,November,December
0,1,290.0,127.0,123.92,154.0,66.0,99.347826,62.153846,99.227273,144.0,156.0,353.000000,345.0
1,2,340.0,189.0,123.92,125.0,67.0,99.000000,49.000000,99.227273,149.0,160.0,397.000000,337.0
2,3,359.0,210.0,136.00,141.0,96.0,101.000000,62.153846,99.227273,141.0,171.0,470.000000,300.0
3,4,318.0,266.0,123.92,89.0,164.0,99.347826,62.153846,99.227273,139.0,175.0,392.000000,276.0
4,5,341.0,227.0,123.92,127.0,152.0,99.347826,76.000000,99.227273,122.0,187.0,296.090909,235.0
